<!--
---
PURPOSE: Scaffold pose project and export frame samples.
REQUIRES:
  - outputs/video/video_assets.parquet
  - outputs/video/frame_times.parquet
PRODUCES:
  - pose_projects/session_{id}/{sleap|dlc}/...
  - outputs/labeling/sleap/{session_id}/{camera}/labels.csv
  - outputs/labeling/sleap/{session_id}/{camera}/frames/*.png
WHAT_NEXT: notebooks/07_Pose_to_Motifs_Feature_Engineering.ipynb
---
-->

# 06 Pose Estimation Setup (SLEAP or DLC)

**Purpose**
Scaffold pose project and export frame samples.

**Requires**
- `outputs/video/video_assets.parquet`
- `outputs/video/frame_times.parquet`

**Produces**
- `pose_projects/session_{id}/{sleap|dlc}/...`
- `outputs/labeling/sleap/{session_id}/{camera}/labels.csv`
- `outputs/labeling/sleap/{session_id}/{camera}/frames/*.png`

**What to run next**
- `notebooks/07_Pose_to_Motifs_Feature_Engineering.ipynb`

We scaffold a pose project and export frame samples with timestamps.


## Environment Setup
We add the repo `src/` to the Python path so notebooks can import shared modules.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()      
ROOT = ROOT.parent              
SRC  = ROOT / "src"           

# put repo + src on sys.path
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


#print("ROOT:", ROOT)
#print("SRC :", SRC, "| exists:", SRC.exists())
#print("sys.path[0:3]:", sys.path[:3])

## Prerequisite Check
We parse the notebook header and validate required artifacts before running downstream steps.

In [ ]:
from pathlib import Path
from reports import parse_notebook_header, validate_prerequisites

nb_path = ROOT / "notebooks" / "06_Pose_Estimation_Setup_SLEAP_or_DLC.ipynb"
header  = parse_notebook_header(nb_path)
missing = validate_prerequisites(header.get("REQUIRES", []))

if missing:
    print("Missing prerequisites:")
    for item in missing:
        print(" -", item)
else:
    print("All prerequisites satisfied.")

## Step 1: Scaffold pose project
This creates the directory structure for your chosen tool.

In [ ]:
import json
from pathlib import Path
from io_sessions import load_sessions_csv, get_session_bundle
from config import get_config
from features_pose import sample_frame_indices, scaffold_pose_project, export_labeling_frames

cfg = get_config()
sessions = load_sessions_csv()
SESSION_IDS = sessions.session_id.tolist()[:1]

for session_id in SESSION_IDS:
    bundle = get_session_bundle(session_id, sessions)
    assets = bundle.load_video_assets()
    frame_times = bundle.load_frame_times()
    if assets is None or assets.empty or frame_times is None or frame_times.empty:
        print("No video assets or frame times available.")
        continue

    project_dir = scaffold_pose_project(session_id, cfg.pose_tool, cfg.pose_projects_dir)
    for camera in assets["camera"].unique():
        row = assets[assets["camera"] == camera].iloc[0]
        video_path = row.get("local_video_path")
        if not video_path:
            print(f"No local video for camera: {camera}")
            continue
        ft_cam = frame_times[frame_times["camera"] == camera]
        if ft_cam.empty:
            print(f"No frame times for camera: {camera}")
            continue
        samples_dir = cfg.outputs_dir / "labeling" / "sleap" / f"{session_id}" / camera
        frame_idx = sample_frame_indices(ft_cam, n_samples=50)
        export_labeling_frames(Path(video_path), frame_idx, samples_dir, ft_cam, session_id, camera)
        print("Labeling frames:", samples_dir)

    print("Pose project:", project_dir)
